In [ ]:
# !git clone https://github.com/alipay/Spatio-Temporal-Hypergraph-Model.git
# %cd Spatio-Temporal-Hypergraph-Model
# !pip install -r requirements.txt

Cloning into 'Spatio-Temporal-Hypergraph-Model'...
remote: Enumerating objects: 140, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 140 (delta 51), reused 119 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (140/140), 32.06 MiB | 17.52 MiB/s, done.
Resolving deltas: 100% (51/51), done.
Error downloading object: data/ca/raw.zip (3ab6e3f): Smudge error: Error downloading data/ca/raw.zip (3ab6e3f7a545f4f3826121aa3a3f84593b8b2ca299fec7293d9727dcb1784c5f): batch response: Git LFS is disabled for this repository.

Errors logged to /content/Spatio-Temporal-Hypergraph-Model/.git/lfs/logs/20250525T183018.215151976.log
Use `git lfs logs last` to view the log.
error: external filter 'git-lfs filter-process' failed
fatal: data/ca/raw.zip: smudge filter lfs failed
You can inspect what was checked out with 'git status'
and retry with 'git restore --source=HEAD :/'

/content/Spatio-Temporal-Hypergraph-Model
     ━━━━━━

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# from drive to colab
!cp -r "/content/drive/My Drive/MSc/Thesis/Next-POI/Baselines/STHGCN" \
      /content/Spatio-Temporal-Hypergraph-Model

In [ ]:
%cd Spatio-Temporal-Hypergraph-Model

/content/Spatio-Temporal-Hypergraph-Model


In [ ]:
import pickle as pkl
from pathlib import Path
import pandas as pd
import os
import glob

In [ ]:
!pip install "numpy<2"

In [ ]:
import numpy as np
np.__version__

'1.26.4'

Restart session, import again and continue...

In [ ]:
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric \
  -f https://data.pyg.org/whl/torch-2.6.0+cu124.html

import torch
print(torch.__version__, torch.version.cuda)
import torch_scatter, torch_sparse, torch_cluster, torch_spline_conv, torch_geometric
print("✔️ PyG imports OK")

Looking in links: https://data.pyg.org/whl/torch-2.6.0+cu124.html
2.6.0+cu124 12.4
✔️ PyG imports OK


In [ ]:
original_path = '/content/drive/My Drive/MSc/Thesis/Next-POI/Forsquare_NYC/Numeric'
base_path =  '/content/drive/My Drive/MSc/Thesis/Next-POI/Baselines/STHGCN'

### Load original trajectories

In [ ]:
with open(os.path.join(original_path, 'train_trajectories.pickle'), 'rb') as f:
    train_trajs = pkl.load(f)

with open(os.path.join(original_path, 'validation_trajectories.pickle'), 'rb') as f:
    validation_trajs = pkl.load(f)

with open(os.path.join(original_path, 'test_trajectories.pickle'), 'rb') as f:
    test_trajs = pkl.load(f)

train_df = pd.read_csv(os.path.join(original_path, 'train_checkins.csv'))
validation_df = pd.read_csv(os.path.join(original_path, 'validation_checkins.csv'))
test_df = pd.read_csv(os.path.join(original_path, 'test_checkins.csv'))

In [ ]:
all_cats = pd.concat(train_trajs + validation_trajs + test_trajs)["poi_category_id"].unique()
cat2code = {cat: i for i,cat in enumerate(sorted(all_cats))}

def dump_sthg(trajectories, out_path):
    rows = []
    for traj in trajectories:
        # traj is a DF of 20 check-ins sorted by local_time
        uid = traj.user_id.iloc[0]
        # use the existing trajectory_id if you set one, else build your own:
        if "trajectory_id" in traj:
            wid = traj.trajectory_id.iloc[0]
        else:
            # fallback: useridx_position
            wid = f"{uid}_{traj.index.min()}"
        first_day = traj.local_time.dt.normalize().iloc[0]

        # now row by row
        for _, row in traj.iterrows():
            lt = row.local_time
            dow = lt.weekday()
            secs = lt.hour*3600 + lt.minute*60 + lt.second
            norm_in_day = secs / (24*3600)
            day_shift = (lt.normalize() - first_day).days
            rows.append({
                "user_id":           uid,
                "POI_id":            row.poi_id,
                "POI_catid":         row.poi_category_id,
                "POI_catid_code":    cat2code[row.poi_category_id],
                "POI_catname":       row.poi_category_name,
                "latitude":          row.latitude,
                "longitude":         row.longitude,
                "timezone":          row.timezone_offset,
                "UTC_time":          row.utc_time,
                "local_time":        lt,
                "day_of_week":       dow,
                "norm_in_day_time":  norm_in_day,
                "trajectory_id":     wid,
                "norm_day_shift":    day_shift,
                # we'll fill norm_relative_time afterwards per‐trajectory
                "_pos_in_traj":      row.name  # temporary, use the DF index or trajectory offset
            })
    df = pd.DataFrame(rows)

    # compute norm_relative_time: within each trajectory 0.0→1.0
    df["pos"] = df.groupby("trajectory_id").cumcount()
    traj_lens = df.groupby("trajectory_id")["pos"].transform("max")
    df["norm_relative_time"] = df["pos"] / traj_lens.astype(float)

    # select & reorder to exactly their 15 columns
    out = df[[
      "user_id","POI_id","POI_catid","POI_catid_code","POI_catname",
      "latitude","longitude","timezone","UTC_time","local_time",
      "day_of_week","norm_in_day_time","trajectory_id",
      "norm_day_shift","norm_relative_time"
    ]]

    # ensure output directory exists
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    # write with no header, tab‐separated
    out.to_csv(out_path, sep="\t", header=True, index=False)

In [ ]:
dump_sthg(train_trajs, os.path.join(base_path, "data/NYC/NYC_train.tsv"))
dump_sthg(validation_trajs, os.path.join(base_path, "data/NYC/NYC_val.tsv"))
dump_sthg(test_trajs, os.path.join(base_path, "data/NYC/NYC_test.tsv"))
# !cp "/content/drive/My Drive/MSc/Thesis/Next-POI/Baselines/STHGCN/data/NYC/NYC_train.tsv"  data/NYC/NYC_train.tsv
# !cp "/content/drive/My Drive/MSc/Thesis/Next-POI/Baselines/STHGCN/data/NYC/NYC_val.tsv"    data/NYC/NYC_val.tsv
# !cp "/content/drive/My Drive/MSc/Thesis/Next-POI/Baselines/STHGCN/data/NYC/NYC_test.tsv"   data/NYC/NYC_test.tsv

In [ ]:
# repo_root = Path("/content/Spatio-Temporal-Hypergraph-Model")
# src_dir = repo_root / "data" / "nyc" / "raw"
# raw_dir = repo_root / "data" / "nyc" / "raw"
# for split in ("train", "val", "test"):
#     tsv = src_dir / f"NYC_{split}.tsv"
#     assert tsv.exists(), f"Missing {tsv}"
#     df = pd.read_csv(tsv, sep="\t", header=0)    # header=0 because you saved with header=True
#     out_csv = raw_dir / f"NYC_{split}.csv"
#     print(f"→ {tsv.name} → {out_csv}")
#     df.to_csv(out_csv, index=False)

## We want every trajectory to be exactly 20 steps

In [ ]:
RAW_DIR = "data/nyc/raw"
OUT_DIR = "data/nyc/raw_len20"
os.makedirs(OUT_DIR, exist_ok=True)

for split in ["train", "val", "test"]:
    df  = pd.read_csv(f"{RAW_DIR}/NYC_{split}.csv")

    # trajectories that have at least 20 records
    long = (df.groupby("trajectory_id")
              .filter(lambda g: len(g) >= 20))

    # sort chronologically *within traj* and keep the **last** 20
    long = (long.sort_values("local_time")
                 .groupby("trajectory_id")
                 .tail(20))

    long.to_csv(f"{OUT_DIR}/NYC_{split}.csv", index=False)

In [ ]:
import torch, torch_geometric
print(torch.__version__, torch.version.cuda, torch.cuda.is_available())

2.0.1+cu117 11.7 True


In [ ]:
# # Clean slate
# !pip uninstall -y torch torchvision torchaudio torchtext torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric

# # Reinstall torch with CUDA (cu117), compatible with PyG wheels
# !pip install --no-cache-dir torch==2.0.1+cu117 torchvision==0.15.2+cu117 torchaudio==2.0.2+cu117 \
#   -f https://download.pytorch.org/whl/torch_stable.html

# # Now install PyG and its extensions compatible with torch 2.0.1 + cu117
# !pip install --no-cache-dir torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric \
#   -f https://data.pyg.org/whl/torch-2.0.1+cu117.html


Found existing installation: torch 2.0.1+cu117
Uninstalling torch-2.0.1+cu117:
  Successfully uninstalled torch-2.0.1+cu117
Found existing installation: torchvision 0.15.2+cu117
Uninstalling torchvision-0.15.2+cu117:
  Successfully uninstalled torchvision-0.15.2+cu117
Found existing installation: torchaudio 2.0.2+cu117
Uninstalling torchaudio-2.0.2+cu117:
  Successfully uninstalled torchaudio-2.0.2+cu117
Found existing installation: torch_scatter 2.1.2+pt26cu124
Uninstalling torch_scatter-2.1.2+pt26cu124:
  Successfully uninstalled torch_scatter-2.1.2+pt26cu124
Found existing installation: torch_sparse 0.6.18+pt26cu124
Uninstalling torch_sparse-0.6.18+pt26cu124:
  Successfully uninstalled torch_sparse-0.6.18+pt26cu124
Found existing installation: torch_cluster 1.6.3+pt26cu124
Uninstalling torch_cluster-1.6.3+pt26cu124:
  Successfully uninstalled torch_cluster-1.6.3+pt26cu124
Found existing installation: torch_spline_conv 1.2.2+pt26cu124
Uninstalling torch_spline_conv-1.2.2+pt26cu124:
 

Looking in links: https://data.pyg.org/whl/torch-2.0.1+cu117.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 91.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 55.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 36.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.1/887.1 kB 13.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.0 MB/s eta 0:00:00


In [ ]:
!python run.py -f best_conf/nyc.yml

2025-05-26 09:29:18.994611: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-26 09:29:19.012044: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748251759.033303   26256 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748251759.039903   26256 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-26 09:29:19.061071: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

evaluating test set!
100% 1/1 [00:00<00:00,  2.65it/s]
Traceback (most recent call last):
  File "/content/Spatio-Temporal-Hypergraph-Model/run.py", line 292, in <module>
    'hparam/NDCG@10': ndcg_res[10],
                      ~~~~~~~~^^^^
KeyError: 10

Copy from colab to drive

In [ ]:
!cp -r /content/Spatio-Temporal-Hypergraph-Model \
       "/content/drive/My Drive/MSc/Thesis/Next-POI/Baselines/STHGCN_2"

In [ ]:
!grep -R "Test evaluation result" log/*/nyc/train.log

log/20250525_185124/nyc/train.log:2025-05-25 19:50:44 INFO     [Evaluating] Test evaluation result : {'hparam/num_params': 27412994, 'hparam/Recall@1': 0.03846153989434242, 'hparam/Recall@5': 0.7307692170143127, 'hparam/Recall@10': 0.7884615659713745, 'hparam/Recall@20': 0.8461538553237915, 'hparam/NDCG@1': 0.03846153989434242, 'hparam/NDCG@5': 0.41743817925453186, 'hparam/NDCG@10': 0.4375486969947815, 'hparam/NDCG@20': 0.45233243703842163, 'hparam/MAP@1': 0.03846153989434242, 'hparam/MAP@5': 0.3121795058250427, 'hparam/MAP@10': 0.3213370144367218, 'hparam/MAP@20': 0.3254985213279724, 'hparam/MRR': 0.3278370499610901}
log/20250525_202347/nyc/train.log:2025-05-25 21:15:33 INFO     [Evaluating] Test evaluation result : {'hparam/num_params': 27412994, 'hparam/Recall@1': 0.057692307978868484, 'hparam/Recall@5': 0.692307710647583, 'hparam/Recall@10': 0.7884615659713745, 'hparam/Recall@20': 0.807692289352417, 'hparam/NDCG@1': 0.057692307978868484, 'hparam/NDCG@5': 0.414340078830719, 'hparam/